With this notebook I aim to analyse what I haven’t yet looked at in all the models we got from radiomics.

We have found scans that are repeated in the lndb and are creating data leakages, but the models we have trained so far have been trained and evaluated with that repetition. So, for now, because its a small number of scans we will leave it as it is, but if we come to train new models, then we will make sure everything can be compared correctly.

**Comparison rule:** the current training default includes these CT copies. All scored runs below used that default. A future run without copies needs `--exclude-duplicate-lndb` and must be evaluated separately. These results are exploratory, not leakage-free estimates.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs/paths.yaml").exists(), "Run from the repository root or notebooks/."
sys.path.insert(0, str(ROOT))

from src.modeling import EXCLUDED_LNDB_IDS, metrics

KEYS = ["LNDbID", "FindingID"]
LIDC_KEYS = ["patient_id", "series_uid", "nodule_id"]
CLASSES = [1, 2, 3, 4, 5]
print("Repository:", ROOT)
print("Duplicate CTs excluded only with --exclude-duplicate-lndb:", sorted(EXCLUDED_LNDB_IDS))

Repository: /home/jp/Documents/por-definir-ds
Duplicate CTs excluded only with --exclude-duplicate-lndb: [181, 182, 183, 184, 185, 188, 189, 194, 195, 198, 226]


## 1. What features do we have?

Training data only. Each row counts nodules with extracted features in that CSV.

In [3]:
feature_files = sorted(
    path for path in (ROOT / "data").glob("*.csv")
    if path.name.startswith((
        "lidc_circle_", "lidc_segmentation_", "lidc_sphere_",
        "lndb_circle_", "lndb_ge3_segmentation_", "lndb_sphere_",
    ))
)

rows = []
for path in feature_files:
    name = path.stem
    dataset = "LIDC" if name.startswith("lidc_") else "LNDb ge3" if name.startswith("lndb_ge3_") else "LNDb"
    keys = LIDC_KEYS if dataset == "LIDC" else KEYS
    roi = "sphere" if "_sphere_" in name else "segmentation" if "_segmentation_" in name else "circle"
    dimension = "3D" if roi == "sphere" or "_3d_" in name else "5 slices" if "_5slice_" in name else "2D"
    nodules = pd.read_csv(path, usecols=keys).drop_duplicates()
    rows.append({"dataset": dataset, "ROI": roi, "dimension": dimension,
                 "CSV": path.name, "nodules": len(nodules)})

feature_table = pd.DataFrame(rows)
display(feature_table.sort_values(["dataset", "ROI", "CSV"]).reset_index(drop=True))

,dataset,ROI,dimension,CSV,nodules
0,LIDC,circle,5 slices,lidc_circle_r11_5slice_hu.csv,2637
1,LIDC,circle,2D,lidc_circle_r11_bw10_hu.csv,2637
2,LIDC,circle,2D,lidc_circle_r11_clipped.csv,2637
3,LIDC,circle,2D,lidc_circle_r11_hu.csv,2637
4,LIDC,circle,2D,lidc_circle_r14_clipped.csv,2637
5,LIDC,circle,2D,lidc_circle_r14_hu.csv,2637
6,LIDC,circle,2D,lidc_circle_r17_clipped.csv,2637
7,LIDC,circle,2D,lidc_circle_r17_hu.csv,2637
8,LIDC,circle,2D,lidc_circle_r21_clipped.csv,2637
9,LIDC,circle,2D,lidc_circle_r21_hu.csv,2637


## 2. What did we change between experiments?

### Circle radii

We start with circles extracted from the original HU values and validate on all 765 LNDb training nodules. Consensus applies to segmentation masks, not circles.

We used the P95, P97 and P99 nodule-size percentiles to choose circle radii for LNDb and LIDC.

| Percentile | LNDb radius | LIDC radius |
|---|---:|---:|
| P95 | 6 mm | 14 mm |
| P97 | 7 mm | 17 mm |
| P99 | 11 mm | 21 mm |

Because nodule sizes differ between the datasets, we tried using each dataset’s radius. We also tried using the same radius in both datasets, taking the LNDb value, to see whether keeping the extraction region the same size works better. We additionally tried 11 mm in LIDC with 6 mm in LNDb.

In [4]:
def display_models(models):
    scores = {}
    for choice, name in models.items():
        table = pd.read_csv(ROOT / "models" / name / "metrics.csv")
        scores[choice] = table.loc[
            table["fold"].eq("overall_oof"),
            ["balanced_accuracy", "macro_f1", "qwk", "mae"],
        ].iloc[0]

    display(pd.DataFrame(scores).T.round(3))

In [5]:
models = {
    "same radius: LNDb 6, LIDC 6": "rf_lidc_circle_r6_hu_lndb_circle_r6_hu_balanced",
    "same radius: LNDb 7, LIDC 7": "rf_lidc_circle_r7_hu_lndb_circle_r7_hu_balanced",
    "same radius: LNDb 11, LIDC 11": "rf_lidc_circle_r11_hu_lndb_circle_r11_hu_balanced",
    "P95: LNDb 6, LIDC 14": "rf_circle_r14_r6_balanced_5class",
    "P97: LNDb 7, LIDC 17": "rf_circle_r17_r7_balanced_5class",
    "P99: LNDb 11, LIDC 21": "rf_lidc_circle_r21_hu_lndb_circle_r11_hu_balanced",
    "extra: LNDb 6, LIDC 11": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
}

display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"same radius: LNDb 6, LIDC 6",0.382,0.383,0.443,0.725
"same radius: LNDb 7, LIDC 7",0.388,0.385,0.445,0.724
"same radius: LNDb 11, LIDC 11",0.360,0.369,0.351,0.790
"P95: LNDb 6, LIDC 14",0.409,0.422,0.435,0.698
"P97: LNDb 7, LIDC 17",0.375,0.388,0.435,0.690
"P99: LNDb 11, LIDC 21",0.345,0.371,0.352,0.736
"extra: LNDb 6, LIDC 11",0.408,0.429,0.442,0.685


We thought P99 might help by including more of the perinodular region, but both P99 choices had lower balanced accuracy than P95. Larger circles may also include nearby vessels and other tissue. These scores cannot tell us why the predictions changed. The P95 14/6 mm model and the extra 11/6 mm model are nearly tied on balanced accuracy (0.409 and 0.408). The 11/6 mm model has lower MAE (0.685 versus 0.698).

### Does clipping help?

As described in the preprocessing notebook, we also extracted features after clipping CT values to [-1000, 400] HU. We compare this with the original HU values using the same 14/6 mm circles, model and 765 LNDb nodules.

In [6]:
clipping_models = {
    "14/6 mm, original HU": "rf_circle_r14_r6_balanced_5class",
    "14/6 mm, clipped HU": "rf_circle_r14_r6_clipped_balanced_5class",
}

display_models(clipping_models)

,balanced_accuracy,macro_f1,qwk,mae
"14/6 mm, original HU",0.409,0.422,0.435,0.698
"14/6 mm, clipped HU",0.356,0.363,0.421,0.707


Clipping did not help this 14/6 mm model. Balanced accuracy fell from 0.409 to 0.356, and MAE rose from 0.698 to 0.707. The original HU values also gave higher macro F1 and QWK here.

### Do circles or 2D segmentations work better?

The radius comparisons above used all 765 LNDb nodules. Segmentation masks are available only for nodules ≥3 mm, so this comparison uses 622 LNDb nodules. We retrained the 14/6 mm circle model on exactly the same LNDb and LIDC nodules as the 2D, 50% consensus segmentation model.

In [7]:
models = {
    "circle, 14/6 mm": "rf_circle_r14_r6_ge3_c50_population_balanced",
    "2D segmentation, 50% consensus": "rf_segmentation_2d_c50_bw25_available_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"circle, 14/6 mm",0.397,0.427,0.414,0.711
"2D segmentation, 50% consensus",0.399,0.414,0.529,0.650


Balanced accuracy is nearly the same (0.397 for circles and 0.399 for segmentation). Segmentation has higher QWK (0.529 versus 0.414) and lower MAE (0.650 versus 0.711), while circles have higher macro F1 (0.427 versus 0.414). Circles use the slice nearest the centroid, whereas segmentation uses the slice with the largest mask area, so the region and selected slice both change.

### What changes with 2D mask consensus?

A higher consensus requires more readers to include a pixel in the mask. With 75% and 100% consensus, some masks become too small to extract features. The 25% and 50% models use 622 LNDb nodules. The 75% and 100% models use 614, so their overall scores also reflect a change in the training and validation population.

In [8]:
models = {
    "25%": "rf_segmentation_2d_c25_bw25_available_balanced",
    "50%": "rf_segmentation_2d_c50_bw25_available_balanced",
    "75%": "rf_segmentation_2d_c75_bw25_available_balanced",
    "100%": "rf_segmentation_2d_c100_bw25_available_balanced",
}
display_models(models)

lndb_ids = []
for level in (75, 100):
    lidc = pd.read_csv(ROOT / "data" / f"lidc_segmentation_2d_c{level}_hu.csv")
    lndb = pd.read_csv(ROOT / "models" / models[f"{level}%"] / "validation_predictions.csv")
    lndb_ids.append(set(zip(lndb.LNDbID, lndb.FindingID)))
    print(f"{level}%: {len(lidc)} LIDC, {len(lndb)} LNDb")

print("Same LNDb nodules:", lndb_ids[0] == lndb_ids[1])

,balanced_accuracy,macro_f1,qwk,mae
25%,0.388,0.399,0.493,0.666
50%,0.399,0.414,0.529,0.650
75%,0.379,0.394,0.503,0.674
100%,0.375,0.388,0.484,0.686


75%: 2630 LIDC, 614 LNDb
100%: 2628 LIDC, 614 LNDb
Same LNDb nodules: True


The 50% consensus model has the highest balanced accuracy, macro F1 and QWK, and the lowest MAE in this table. The 75% and 100% models also lose eight LNDb nodules and some LIDC nodules during extraction. Their score differences cannot be attributed to the consensus level alone.

### Does 3D segmentation help?

Both models use 50% consensus masks and validate on the same 622 LNDb nodules. The 2D model extracts features from one axial slice, while the 3D model uses the full mask volume. The 3D extraction also includes one more LIDC training nodule.

In [9]:
models = {
    "2D": "rf_segmentation_2d_c50_bw25_available_balanced",
    "3D": "rf_segmentation_3d_c50_bw25_ge3_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
2D,0.399,0.414,0.529,0.650
3D,0.428,0.446,0.523,0.632


The 3D model has higher balanced accuracy (0.428 versus 0.399) and macro F1 (0.446 versus 0.414), and lower MAE (0.632 versus 0.650). The 2D model has slightly higher QWK (0.529 versus 0.523). Since the LIDC training populations differ by one nodule, this does not isolate the effect of 2D versus 3D extraction completely.

### Does a 3D sphere help?

At 11/6 mm, compare a circle on one slice with a 3D sphere around the same centroid. Both models use all 765 LNDb training nodules.

In [18]:
models = {
    "2D circle, 11/6 mm": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
    "3D sphere, 11/6 mm": "rf_sphere_bw25_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"2D circle, 11/6 mm",0.408,0.429,0.442,0.685
"3D sphere, 11/6 mm",0.399,0.416,0.489,0.655


The sphere has higher QWK (0.489 versus 0.442) and lower MAE (0.655 versus 0.685), but lower balanced accuracy (0.399 versus 0.408).

### Do spheres or 3D segmentations work better?

To compare a sphere with the 3D, 50% consensus segmentation, restrict the sphere model to the same 622 LNDb nodules. The full-population sphere score above is not directly comparable with this segmentation score.

In [19]:
models = {
    "3D sphere, LNDb ≥3 mm": "rf_sphere_bw25_ge3_balanced",
    "3D segmentation, 50%": "rf_segmentation_3d_c50_bw25_ge3_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"3D sphere, LNDb ≥3 mm",0.401,0.428,0.475,0.680
"3D segmentation, 50%",0.428,0.446,0.523,0.632


On these 622 nodules, 3D segmentation improves balanced accuracy (0.428 versus 0.401), QWK (0.523 versus 0.475) and MAE (0.632 versus 0.680) over the sphere. Unlike the sphere, it uses the provided nodule mask.

### Do five circle slices help?

We also trained a five-class RF using five adjacent circle slices instead of one. Each training nodule contributes five rows, although validation still counts each nodule once.

In [20]:
models = {
    "one circle slice, 11/6 mm": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
    "five circle slices, 11/6 mm": "rf_circle_r11_r6_5slice_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"one circle slice, 11/6 mm",0.408,0.429,0.442,0.685
"five circle slices, 11/6 mm",0.353,0.373,0.396,0.708


For five-class prediction, five slices reduce balanced accuracy from 0.408 to 0.353 and QWK from 0.442 to 0.396. The binary result is a separate question below.

### Does a smaller bin width help?

As described in the radiomics decisions notebook, bin width groups CT values for texture features. Here we compare 25 HU with 10 HU using the same 11/6 mm circles and 765 LNDb nodules.

In [10]:
models = {
    "25 HU": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
    "10 HU": "rf_circle_r11_r6_bw10_balanced",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
25 HU,0.408,0.429,0.442,0.685
10 HU,0.370,0.382,0.437,0.695


For these 11/6 mm circles, 25 HU gives higher balanced accuracy (0.408 versus 0.370), macro F1 and QWK, and lower MAE (0.685 versus 0.695) than 10 HU. This comparison does not establish the best bin width for other regions or radii.

## 3. Does training on LIDC or changing class balance help?

We keep the LNDb circle radius at 6 mm and compare training on LNDb alone with training on LNDb plus LIDC, using 14 mm circles for LIDC. Both models are evaluated on the same 765 LNDb nodules.

In [11]:
models = {
    "LNDb only, 6 mm": "rf_lndb_circle_r6_hu_balanced",
    "LNDb + LIDC, 14/6 mm": "rf_circle_r14_r6_balanced_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"LNDb only, 6 mm",0.409,0.430,0.446,0.681
"LNDb + LIDC, 14/6 mm",0.409,0.422,0.435,0.698


No big difference.


The five-class labels are uneven. Among the 765 LNDb nodules, 254 are class 2 but only 18 are class 5. We first compare RF with and without class weights, then class weights, SMOTE and Balanced RF on the same 14/6 mm circles.

For 3D spheres and 50% consensus segmentations, compare the three balancing methods separately on the same 622 LNDb nodules ≥3 mm. The sphere results here use the ≥3 mm model, not the 765-nodule sphere model.

In [12]:
models = {
    "no class weights": "rf_circle_r14_r6_unweighted_5class",
    "balanced class weights": "rf_circle_r14_r6_balanced_5class",
}
display_models(models)
models = {
    "LNDb only, no class weights": "rf_lndb_circle_r6_hu",
    "LNDb only, balanced": "rf_lndb_circle_r6_hu_balanced",
}
display_models(models)
models = {
    "class weights": "rf_circle_r14_r6_balanced_5class",
    "SMOTE": "rf_circle_r14_r6_smote_5class",
    "balanced RF": "brf_circle_r14_r6_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
no class weights,0.385,0.402,0.467,0.667
balanced class weights,0.409,0.422,0.435,0.698


,balanced_accuracy,macro_f1,qwk,mae
"LNDb only, no class weights",0.379,0.393,0.458,0.672
"LNDb only, balanced",0.409,0.430,0.446,0.681


,balanced_accuracy,macro_f1,qwk,mae
class weights,0.409,0.422,0.435,0.698
SMOTE,0.412,0.399,0.439,0.736
balanced RF,0.426,0.323,0.414,0.809


In [ ]:
models = {
    "class weights, 3D sphere": "rf_sphere_bw25_ge3_balanced",
    "SMOTE, 3D sphere": "rf_sphere_ge3_smote_5class",
    "Balanced RF, 3D sphere": "brf_sphere_ge3_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"class weights, 3D sphere",0.401,0.428,0.475,0.680
"SMOTE, 3D sphere",0.452,0.436,0.500,0.722
"Balanced RF, 3D sphere",0.484,0.385,0.477,0.788


In [ ]:
models = {
    "class weights, 3D segmentation": "rf_segmentation_3d_c50_bw25_ge3_balanced",
    "SMOTE, 3D segmentation": "rf_segmentation_3d_c50_ge3_smote_5class",
    "Balanced RF, 3D segmentation": "brf_segmentation_3d_c50_ge3_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"class weights, 3D segmentation",0.428,0.446,0.523,0.632
"SMOTE, 3D segmentation",0.445,0.435,0.504,0.698
"Balanced RF, 3D segmentation",0.480,0.416,0.519,0.743


With 14/6 mm circles, Balanced RF has the highest balanced accuracy among these three balancing methods, but class-weighted RF has higher macro F1 and lower MAE. On both 3D regions, Balanced RF again has the highest balanced accuracy (0.484 for spheres and 0.480 for segmentations), while class-weighted RF has lower MAE (0.680 and 0.632). No balancing method wins on every metric.

## 4. Do fewer features help?

First compare all 14/6 mm circle features with top 100, top 500, Original-only and normalized pairs for each balancing method. Original-only uses features from the CT image without additional filters. Normalized pairs keeps other features, but removes a measure's non-normalized version when its `Normalized` version exists.

Then compare all features with top 100, 500 and 1000 for class-weighted RF on 11/6 mm 3D spheres and 50% consensus 3D segmentations. Balanced RF feature selection is also shown where trained. The 3D models use the same 622 LNDb nodules ≥3 mm, while circles use 765 nodules. Top-k uses [ANOVA F scores](https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.f_classif.html) and is selected inside each training fold.

In [27]:
models = {
    "all, class weights": "rf_circle_r14_r6_balanced_5class",
    "top 100, class weights": "rf_circle_r14_r6_balanced_k100_5class",
    "top 500, class weights": "rf_circle_r14_r6_balanced_k500_5class",
    "Original only, class weights": "rf_circle_r14_r6_balanced_original_5class",
    "normalized pairs, class weights": "rf_circle_r14_r6_balanced_normalized_pairs_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, class weights",0.409,0.422,0.435,0.698
"top 100, class weights",0.355,0.370,0.428,0.697
"top 500, class weights",0.396,0.416,0.432,0.699
"Original only, class weights",0.362,0.382,0.434,0.718
"normalized pairs, class weights",0.395,0.425,0.429,0.693


In [ ]:
models = {
    "all, SMOTE": "rf_circle_r14_r6_smote_5class",
    "top 100, SMOTE": "rf_circle_r14_r6_smote_k100_5class",
    "top 500, SMOTE": "rf_circle_r14_r6_smote_k500_5class",
    "Original only, SMOTE": "rf_circle_r14_r6_smote_original_5class",
    "normalized pairs, SMOTE": "rf_circle_r14_r6_smote_normalized_pairs_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, SMOTE",0.412,0.399,0.439,0.736
"top 100, SMOTE",0.340,0.323,0.424,0.766
"top 500, SMOTE",0.409,0.389,0.433,0.749
"Original only, SMOTE",0.359,0.329,0.403,0.841
"normalized pairs, SMOTE",0.405,0.396,0.430,0.750


In [ ]:
models = {
    "all, Balanced RF": "brf_circle_r14_r6_5class",
    "top 100, Balanced RF": "brf_circle_r14_r6_k100_5class",
    "top 500, Balanced RF": "brf_circle_r14_r6_k500_5class",
    "Original only, Balanced RF": "brf_circle_r14_r6_original_5class",
    "normalized pairs, Balanced RF": "brf_circle_r14_r6_normalized_pairs_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, Balanced RF",0.426,0.323,0.414,0.809
"top 100, Balanced RF",0.420,0.334,0.435,0.783
"top 500, Balanced RF",0.417,0.313,0.402,0.825
"Original only, Balanced RF",0.424,0.315,0.390,0.843
"normalized pairs, Balanced RF",0.434,0.332,0.419,0.799


In [ ]:
models = {
    "all, 3D sphere": "rf_sphere_bw25_ge3_balanced",
    "top 100, 3D sphere": "rf_sphere_ge3_balanced_k100_5class",
    "top 500, 3D sphere": "rf_sphere_ge3_balanced_k500_5class",
    "top 1000, 3D sphere": "rf_sphere_ge3_balanced_k1000_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, 3D sphere",0.401,0.428,0.475,0.680
"top 100, 3D sphere",0.414,0.443,0.485,0.670
"top 500, 3D sphere",0.410,0.432,0.492,0.677
"top 1000, 3D sphere",0.380,0.401,0.456,0.704


In [ ]:
models = {
    "all, 3D sphere Balanced RF": "brf_sphere_ge3_5class",
    "top 100, 3D sphere Balanced RF": "brf_sphere_ge3_k100_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, 3D sphere Balanced RF",0.484,0.385,0.477,0.788
"top 100, 3D sphere Balanced RF",0.466,0.387,0.482,0.778


In [ ]:
models = {
    "all, 3D segmentation RF": "rf_segmentation_3d_c50_bw25_ge3_balanced",
    "top 100, 3D segmentation RF": "rf_segmentation_3d_c50_ge3_balanced_k100_5class",
    "top 500, 3D segmentation RF": "rf_segmentation_3d_c50_ge3_balanced_k500_5class",
    "top 1000, 3D segmentation RF": "rf_segmentation_3d_c50_ge3_balanced_k1000_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, 3D segmentation RF",0.428,0.446,0.523,0.632
"top 100, 3D segmentation RF",0.425,0.447,0.504,0.638
"top 500, 3D segmentation RF",0.412,0.430,0.495,0.662
"top 1000, 3D segmentation RF",0.409,0.426,0.504,0.656


In [ ]:
models = {
    "all, 3D segmentation Balanced RF": "brf_segmentation_3d_c50_ge3_5class",
    "top 100, 3D segmentation Balanced RF": "brf_segmentation_3d_c50_ge3_k100_5class",
    "top 500, 3D segmentation Balanced RF": "brf_segmentation_3d_c50_ge3_k500_5class",
    "top 1000, 3D segmentation Balanced RF": "brf_segmentation_3d_c50_ge3_k1000_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"all, 3D segmentation Balanced RF",0.480,0.416,0.519,0.743
"top 100, 3D segmentation Balanced RF",0.456,0.415,0.520,0.719
"top 500, 3D segmentation Balanced RF",0.446,0.400,0.505,0.759
"top 1000, 3D segmentation Balanced RF",0.468,0.411,0.515,0.752


With circles, top-k selection does not improve balanced accuracy for class-weighted RF or SMOTE. Balanced RF improves from 0.426 to 0.434 with normalized pairs, but still has higher MAE than class-weighted RF.

For 3D spheres, class-weighted RF improves across all four metrics with top 100 (balanced accuracy 0.414 versus 0.401 with all features). For 3D segmentation, all features have the highest balanced accuracy within both RF settings (0.428 with class weights and 0.480 with Balanced RF). Feature reduction helps the sphere RF, but not the 3D segmentation models on balanced accuracy.

## 5. How do Balanced RF and XGBoost compare?

Compare both models using all features within each region: 14/6 mm circles on 765 LNDb nodules, then 11/6 mm 3D spheres and 50% consensus 3D segmentations on the same 622 LNDb nodules ≥3 mm. Balanced RF undersamples each tree, while XGBoost uses balanced sample weights, so this tests the complete model settings rather than the algorithm alone.

In [ ]:
models = {
    "Balanced RF, 14/6 mm circles": "brf_circle_r14_r6_5class",
    "XGBoost, 14/6 mm circles": "xgboost_circle_r14_r6_balanced_5class",
}
display_models(models)

models = {
    "Balanced RF, 3D sphere": "brf_sphere_ge3_5class",
    "XGBoost, 3D sphere": "xgboost_sphere_ge3_balanced_5class",
}
display_models(models)

models = {
    "Balanced RF, 3D segmentation": "brf_segmentation_3d_c50_ge3_5class",
    "XGBoost, 3D segmentation": "xgboost_segmentation_3d_c50_ge3_balanced_5class",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
"Balanced RF, 14/6 mm circles",0.426,0.323,0.414,0.809
"XGBoost, 14/6 mm circles",0.436,0.386,0.489,0.718


,balanced_accuracy,macro_f1,qwk,mae
"Balanced RF, 3D sphere",0.484,0.385,0.477,0.788
"XGBoost, 3D sphere",0.442,0.410,0.502,0.725


,balanced_accuracy,macro_f1,qwk,mae
"Balanced RF, 3D segmentation",0.480,0.416,0.519,0.743
"XGBoost, 3D segmentation",0.455,0.430,0.529,0.712


With circles, XGBoost outperforms Balanced RF on all four metrics. With 3D spheres and segmentations, Balanced RF has higher balanced accuracy (0.484 and 0.480 versus 0.442 and 0.455), while XGBoost has higher macro F1 and QWK and lower MAE. These are matched comparisons within each region, not across the 765- and 622-nodule populations.

### Earlier RF tuning at 11/6 mm

The saved comparison below tunes ordinary RF on 11/6 mm circles and also shows a separate normalized-pairs feature test. It does not tune the 14/6 mm or 3D Balanced RF and XGBoost models.

In [17]:
models = {
    "ordinary RF": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced",
    "tuned RF": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced_tuned",
    "normalized pairs": "rf_lidc_circle_r11_hu_lndb_circle_r6_hu_balanced_normalized_pairs",
}
display_models(models)

,balanced_accuracy,macro_f1,qwk,mae
ordinary RF,0.408,0.429,0.442,0.685
tuned RF,0.406,0.414,0.447,0.686
normalized pairs,0.404,0.433,0.433,0.685


Tuning gives almost the same balanced accuracy as ordinary RF (0.406 versus 0.408), with a small QWK increase (0.447 versus 0.442). Normalized pairs give slightly higher macro F1 (0.433 versus 0.429) but lower balanced accuracy and QWK. MAE is 0.685 for both ordinary RF and normalized pairs.

## 6. Which high-rated nodules do the models miss?

Evaluate every model below on the **same 471 LNDb nodules** with saved validation predictions. Of these, 70 have reader-mean rating 4, 18 have rating 5, and 380 have a mean below 3. Three more have a mean above 3 but round to rating 3; they count in neither the rating-4/5 recall nor the low-rating specificity. These are radiologist ratings, not confirmed cancer.

An alert means prediction 4–5 for a five-class model or prediction 1 for a binary model. **Recall (sensitivity)** measures alerts among the 88 rating-4/5 nodules. **Specificity** measures how often the 380 lower-rated nodules avoid an alert. The five-class models also trained on mean-3 nodules excluded from binary training.

### Five-class models

These five-class models validate on at least 622 nodules, but the table uses only the 471 shared nodules. The 2D and 3D segmentations use 50% consensus masks. Only ordinary RF has a saved 2D segmentation model. Its circle model was trained on the same population as the 2D segmentation model; Balanced RF and XGBoost circle models trained on a larger population. The region differences do not isolate the effect of extraction alone.

In [ ]:
five_class_models = {
    ("RF, class weights", "circle (14/6 mm)"): "rf_circle_r14_r6_ge3_c50_population_balanced",
    ("RF, class weights", "2D segmentation (50%)"): "rf_segmentation_2d_c50_bw25_available_balanced",
    ("RF, class weights", "3D sphere (11/6 mm)"): "rf_sphere_bw25_ge3_balanced",
    ("RF, class weights", "3D segmentation (50%)"): "rf_segmentation_3d_c50_bw25_ge3_balanced",
    ("Balanced RF", "circle (14/6 mm)"): "brf_circle_r14_r6_5class",
    ("Balanced RF", "3D sphere (11/6 mm)"): "brf_sphere_ge3_5class",
    ("Balanced RF", "3D segmentation (50%)"): "brf_segmentation_3d_c50_ge3_5class",
    ("XGBoost, class weights", "circle (14/6 mm)"): "xgboost_circle_r14_r6_balanced_5class",
    ("XGBoost, class weights", "3D sphere (11/6 mm)"): "xgboost_sphere_ge3_balanced_5class",
    ("XGBoost, class weights", "3D segmentation (50%)"): "xgboost_segmentation_3d_c50_ge3_balanced_5class",
}
binary_models = {
    ("RF, class weights", "circle (14/6 mm)"): "rf_circle_r14_r6_balanced_binary",
    ("RF, class weights", "3D segmentation (50%)"): "rf_segmentation_3d_c50_ge3_balanced_binary",
    ("Balanced RF", "circle (14/6 mm)"): "brf_circle_r14_r6_binary",
    ("Balanced RF", "3D segmentation (50%)"): "brf_segmentation_3d_c50_ge3_binary",
    ("XGBoost, class weights", "circle (14/6 mm)"): "xgboost_circle_r14_r6_balanced_binary",
    ("XGBoost, class weights", "3D segmentation (50%)"): "xgboost_segmentation_3d_c50_ge3_balanced_binary",
}

In [ ]:
keys = KEYS + ["fold"]
five_class_validation = pd.read_csv(
    ROOT / "models" / five_class_models[("RF, class weights", "3D segmentation (50%)")]
    / "validation_predictions.csv"
)[keys + ["target_classification_mean"]]
binary_validation = pd.read_csv(
    ROOT / "models" / binary_models[("RF, class weights", "3D segmentation (50%)")]
    / "validation_predictions.csv"
)[keys + ["target_binary_mean"]].merge(five_class_validation, on=keys, validate="one_to_one")
assert len(five_class_validation) == 622 and len(binary_validation) == 471

def alert_table(models, binary=False):
    target = "target_binary_mean" if binary else "target_classification_mean"
    rows = []
    for (model, region), name in models.items():
        predictions = pd.read_csv(ROOT / "models" / name / "validation_predictions.csv")
        shared = binary_validation.merge(
            predictions, on=keys, how="left", suffixes=("", "_saved"), validate="one_to_one"
        )
        assert len(shared) == 471 and shared.prediction.notna().all()
        assert shared[target].eq(shared[f"{target}_saved"]).all()
        class_4 = shared.target_classification_mean.eq(4)
        class_5 = shared.target_classification_mean.eq(5)
        low = shared.target_binary_mean.eq(0)
        alert = shared.prediction.eq(1) if binary else shared.prediction.ge(4)
        missed_4 = int((class_4 & ~alert).sum())
        missed_5 = int((class_5 & ~alert).sum())
        false_alerts = int((low & alert).sum())
        assert (class_4.sum(), class_5.sum(), low.sum()) == (70, 18, 380)
        rows.append({
            "model": model, "region": region,
            "missed rating 4 / 70": missed_4,
            "missed rating 5 / 18": missed_5,
            "missed ratings 4–5 / 88": missed_4 + missed_5,
            "rating 4–5 recall": 1 - (missed_4 + missed_5) / 88,
            "specificity": 1 - false_alerts / 380,
            "false alerts / 380": false_alerts,
        })
    return pd.DataFrame(rows).set_index(["model", "region"]).round(3)

display(alert_table(five_class_models))

missed rating 4 / 70  missed rating 5 / 18  missed ratings 4–5 / 88  rating 4–5 recall  specificity  false alerts / 380
model                  region                                                                                                                                        
RF, class weights      circle (14/6 mm)                         58                     7                       65              0.261        0.979                   8
                       2D segmentation (50%)                    45                     5                       50              0.432        0.955                  17
                       3D sphere (11/6 mm)                      53                     6                       59              0.330        0.963                  14
                       3D segmentation (50%)                    47                     6                       53              0.398        0.955                  17
Balanced RF            circle (14/6 mm)                         48                     3                       51              0.420        0.900                  38
                       3D sphere (11/6 mm)                      32                     3                       35              0.602        0.892                  41
                       3D segmentation (50%)                    23                     2                       25              0.716        0.866                  51
XGBoost, class weights circle (14/6 mm)                         42                     3                       45              0.489        0.924                  29
                       3D sphere (11/6 mm)                      34                     4                       38              0.568        0.911                  34
                       3D segmentation (50%)                    31                     3                       34              0.614        0.879                  46

Five-class 3D segmentation Balanced RF misses the fewest high-rated nodules (25/88, including 2/18 rated 5). It alerts on 63/88 versus 53/88 with a 3D sphere and 37/88 with circles, but raises 51 versus 41 and 38 false alerts among the 380 lower-rated nodules. For RF, 2D segmentation alerts on 38/88 versus 23/88 with its population-matched circle model, with 17 versus 8 false alerts.

### Binary models

The binary target is 0 for a reader mean below 3 and 1 for a mean above 3; means of exactly 3 are excluded. Only circles and 3D segmentation have saved binary models. The circle models trained on more nodules than the segmentation models, even though every row below is evaluated on the same 471 nodules. Confirmed duplicate LNDb CTs were not excluded.

In [ ]:
display(alert_table(binary_models, binary=True))

missed rating 4 / 70  missed rating 5 / 18  missed ratings 4–5 / 88  rating 4–5 recall  specificity  false alerts / 380
model                  region                                                                                                                                        
RF, class weights      circle (14/6 mm)                         61                     7                       68              0.227        0.971                  11
                       3D segmentation (50%)                    45                     4                       49              0.443        0.921                  30
Balanced RF            circle (14/6 mm)                         47                     4                       51              0.420        0.926                  28
                       3D segmentation (50%)                    30                     4                       34              0.614        0.861                  53
XGBoost, class weights circle (14/6 mm)                         43                     5                       48              0.455        0.918                  31
                       3D segmentation (50%)                    31                     2                       33              0.625        0.871                  49

With binary Balanced RF on these shared nodules, 3D segmentation alerts on 54/88 rating-4/5 nodules versus 37/88 with circles. It also raises more false alerts on binary-0 nodules (53 versus 28), so recall improves while specificity falls. Recall is what matters the most in the clinic context.

## 7. Does regression provide useful alerts?

Ridge predicts the **unrounded radiologist mean**. Compare the three regions on the same 622 LNDb nodules using continuous MAE/RMSE, rounded five-class QWK, and signed error on true ratings 4–5. A negative signed error means systematic underprediction. Continuous error uses raw predictions, even when they fall outside 1–5. The earlier 11/6 mm circle run covers 765 nodules and is kept separate.

For alerts, restrict all three models to the **same 471 nodules as section 6**. The saved five-class prediction rounds the continuous value to 1–5 (an alert starts at **3.5**). A derived binary prediction alerts when the continuous value is **above 3**. Both thresholds are fixed, not chosen using validation outcomes. Of the 471, 88 have rating 4–5, 91 have an unrounded mean above 3, and 380 have a mean below 3. Three binary-1 nodules round to rating 3. These are reader ratings, not confirmed diagnoses.

In [ ]:
from src.regression import regression_metrics

ridge_models = {
    "circle (14/6 mm)": "ridge_circle_r14_r6_ge3",
    "3D sphere (11/6 mm)": "ridge_sphere_ge3",
    "3D segmentation (50%)": "ridge_segmentation_3d_c50_ge3",
}
targets = pd.read_csv(ROOT / "data/processed/lndb_ge3_nodule_targets.csv")
binary_targets = targets.rename(columns={"NoduleID": "FindingID"})[
    KEYS + ["target_binary_mean"]
]
regression_rows, alert_rows = [], []
for region, name in ridge_models.items():
    predictions = pd.read_csv(ROOT / "models" / name / "validation_predictions.csv")
    assert len(predictions) == 622
    assert predictions.prediction.eq(
        np.clip(np.floor(predictions.prediction_continuous + 0.5), 1, 5)
    ).all()
    high = predictions.target_classification_mean.ge(4)
    scores = regression_metrics(predictions.target_regression_mean, predictions.prediction_continuous)
    regression_rows.append({
        "region": region,
        "continuous MAE": scores["mae"],
        "continuous RMSE": scores["rmse"],
        "bias on ratings 4–5": (predictions.loc[high, "prediction_continuous"]
                                - predictions.loc[high, "target_regression_mean"]).mean(),
        "predictions outside 1–5": int((~predictions.prediction_continuous.between(1, 5)).sum()),
        "rounded five-class QWK": metrics(predictions.target_classification_mean,
                                          predictions.prediction)["qwk"],
    })
    shared = predictions.merge(binary_targets, on=KEYS, validate="one_to_one")
    shared = shared.loc[shared.target_binary_mean.notna()].copy()
    rating_4 = shared.target_classification_mean.eq(4)
    rating_5 = shared.target_classification_mean.eq(5)
    binary_1 = shared.target_binary_mean.eq(1)
    binary_0 = shared.target_binary_mean.eq(0)
    assert len(shared) == 471 and (rating_4.sum(), rating_5.sum()) == (70, 18)
    assert (binary_1.sum(), binary_0.sum()) == (91, 380)
    assert binary_1.eq(shared.target_regression_mean.gt(3)).all()
    for rule, alert in {
        "rounded 1–5 (alert 4–5)": shared.prediction.ge(4),
        "binary (mean >3)": shared.prediction_continuous.gt(3),
    }.items():
        missed_4 = int((rating_4 & ~alert).sum())
        missed_5 = int((rating_5 & ~alert).sum())
        false_alerts = int((binary_0 & alert).sum())
        alert_rows.append({
            "region": region, "alert rule": rule,
            "missed rating 4 / 70": missed_4,
            "missed rating 5 / 18": missed_5,
            "missed ratings 4–5 / 88": missed_4 + missed_5,
            "rating 4–5 recall": 1 - (missed_4 + missed_5) / 88,
            "binary-1 recall": int((binary_1 & alert).sum()) / 91,
            "specificity": 1 - false_alerts / 380,
            "false alerts / 380": false_alerts,
        })
display(pd.DataFrame(regression_rows).set_index("region").round(3))
display(pd.DataFrame(alert_rows).set_index(["region", "alert rule"]).round(3))

,continuous MAE,continuous RMSE,bias on ratings 4–5,predictions outside 1–5,rounded five-class QWK
region,,,,,
circle (14/6 mm),0.805,1.053,-1.150,35,0.378
3D sphere (11/6 mm),0.916,2.175,-1.488,59,0.418
3D segmentation (50%),0.821,1.140,-1.094,52,0.427


missed rating 4 / 70  missed rating 5 / 18  missed ratings 4–5 / 88  rating 4–5 recall  binary-1 recall  specificity  false alerts / 380
region                alert rule                                                                                                                                                       
circle (14/6 mm)      rounded 1–5 (alert 4–5)                    53                    11                       64              0.273            0.286        0.968                  12
                      binary (mean >3)                           44                     5                       49              0.443            0.451        0.866                  51
3D sphere (11/6 mm)   rounded 1–5 (alert 4–5)                    54                     9                       63              0.284            0.286        0.932                  26
                      binary (mean >3)                           37                     6                       43              0.511            0.505        0.850                  57
3D segmentation (50%) rounded 1–5 (alert 4–5)                    51                     9                       60              0.318            0.308        0.979                   8
                      binary (mean >3)                           33                     5                       38              0.568            0.571        0.845                  59

The main concern is missed nodules rated 4–5. On the same nodules, five-class Ridge misses 60/88, compared with 25/88 for five-class Balanced RF. Binary Ridge misses 38/88, compared with 33/88 for binary XGBoost. Ridge therefore does not reduce the most important error. Its five-class conversion has fewer false positives (8 versus 51), but its binary conversion has more (59 versus 49). These are radiologist ratings, not confirmed cancer diagnoses.